In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install pdfplumber python-docx jiwer openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.9 MB/s eta 0:00:00


In [4]:
import os
import re
import tempfile
import pandas as pd
from dataclasses import dataclass
from typing import List

import pdfplumber
from docx import Document
from jiwer import wer, cer

In [5]:
# =========================================================
# Data Class
# =========================================================

@dataclass
class ExtractedDocument:
    title: str = ""
    abstract: str = ""
    keywords: str = ""
    full_text: str = ""


# =========================================================
# Academic Document Extractor (Simplified)
# =========================================================

class AcademicDocumentExtractor:
    """
    Simple & robust academic document extractor.
    Focus:
    - Title
    - Abstract
    - Keywords
    """

    SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".txt"}

    ABSTRACT_PATTERN = re.compile(r"^abstract\s*:?\s*$", re.IGNORECASE)
    KEYWORDS_PATTERN = re.compile(
        r"^(keywords?|key\s+words?|kata\s+kunci)\s*:?\s*",
        re.IGNORECASE
    )

    ABSTRACT_END_MARKERS = (
        "keywords",
        "key words",
        "introduction",
        "1.",
        "i.",
        "doi",
        "http",
        "www",
        "©",
        "copyright",
    )

    # -----------------------------------------------------
    # Public API
    # -----------------------------------------------------

    def extract(self, file_bytes: bytes, filename: str) -> ExtractedDocument:
        if not filename or not file_bytes:
            raise ValueError("Invalid file input")

        ext = os.path.splitext(filename.lower())[1]
        if ext not in self.SUPPORTED_EXTENSIONS:
            raise ValueError(f"Unsupported file type: {ext}")

        lines = self._read_lines(file_bytes, ext)

        document = ExtractedDocument()
        document.full_text = "\n".join(lines)

        document.title = self._extract_title(lines)
        document.abstract = self._extract_abstract(lines)
        document.keywords = self._extract_keywords(lines)

        return document

    # -----------------------------------------------------
    # File Readers
    # -----------------------------------------------------

    def _read_lines(self, file_bytes: bytes, ext: str) -> List[str]:
        if ext == ".pdf":
            return self._read_pdf(file_bytes)
        if ext == ".docx":
            return self._read_docx(file_bytes)
        return self._read_text(file_bytes)

    def _read_pdf(self, file_bytes: bytes) -> List[str]:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
            tmp.write(file_bytes)
            path = tmp.name

        try:
            lines = []
            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        lines.extend(
                            line.strip()
                            for line in text.split("\n")
                            if line.strip()
                        )
            return lines
        finally:
            try:
                os.unlink(path)
            except OSError:
                pass

    def _read_docx(self, file_bytes: bytes) -> List[str]:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".docx") as tmp:
            tmp.write(file_bytes)
            path = tmp.name

        try:
            doc = Document(path)
            return [
                para.text.strip()
                for para in doc.paragraphs
                if para.text.strip()
            ]
        finally:
            try:
                os.unlink(path)
            except OSError:
                pass

    def _read_text(self, file_bytes: bytes) -> List[str]:
        for encoding in ("utf-8", "latin-1", "cp1252"):
            try:
                return [
                    line.strip()
                    for line in file_bytes.decode(encoding).splitlines()
                    if line.strip()
                ]
            except UnicodeDecodeError:
                continue
        return []

    # -----------------------------------------------------
    # Extraction Logic
    # -----------------------------------------------------

    def _extract_title(self, lines: List[str]) -> str:
        """
        Title biasanya:
        - berada di awal dokumen
        - tidak terlalu pendek / panjang
        - bukan all lowercase
        """
        for line in lines[:5]:
            if 10 < len(line) < 200 and not line.islower():
                return re.sub(r"^\d+[.\)]\s*", "", line).strip()
        return ""

    def _extract_abstract(self, lines: List[str]) -> str:
        buffer = []
        in_abstract = False

        for line in lines:
            lower = line.lower()

            if self.ABSTRACT_PATTERN.match(line):
                in_abstract = True
                continue

            if in_abstract:
                if (
                    self.KEYWORDS_PATTERN.match(line)
                    or any(marker in lower for marker in self.ABSTRACT_END_MARKERS)
                ):
                    break
                buffer.append(line)

        return " ".join(buffer).strip()

    def _extract_keywords(self, lines: List[str]) -> str:
        for line in lines:
            if self.KEYWORDS_PATTERN.match(line):
                keywords = self.KEYWORDS_PATTERN.sub("", line)
                keywords = re.sub(r"[;,\s]+", ", ", keywords)
                return keywords.strip(", ")
        return ""


# =========================================================
# Backward Compatibility Alias
# =========================================================

DocumentExtractor = AcademicDocumentExtractor

In [6]:
def normalize(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [8]:
GT_PATH = "/content/ground truth information extraction.xlsx"
df_gt = pd.read_excel(GT_PATH)

df_gt["id"] = df_gt["id"].astype(str)
df_gt.head()

,id,title_gt,abstract_gt,keyword_gt
0,1,"Cancer Incidence, Mortality, Years of Life Los...","IMPORTANCE The Global Burden of Diseases, Inju...",NaN
1,2,Antibacterial Pathways in Transition Metal-Bas...,"Across the planet, outbreaks of bacterial illn...","transition metals, nanocomposites, photodynami..."
2,3,Hypertension is associated with antibody respo...,Several types of vaccines have been developed ...,"COVID-19, Inactivated viral vaccine, Antibody ..."
3,4,Preparation of antibacterial Gel/PCL nanofiber...,"In this work, blend nanofibrous scaffolds were...","Bone tissue engineering, DCPD-reinforced, Cont..."
4,5,"Occurrence, potential sources and ecological r...",Extensive global plastic production has led to...,"Microplastic, Marine ecosystem, Surface water,..."


In [9]:
PDF_FOLDER = "/content/drive/MyDrive/wer cer"

In [13]:
extractor = AcademicDocumentExtractor()

results = []

for _, row in df_gt.iterrows():
    doc_id = row["id"]
    pdf_path = os.path.join(PDF_FOLDER, f"{doc_id}.pdf")

    if not os.path.exists(pdf_path):
        print(f"❌ File tidak ditemukan: {doc_id}.pdf")
        continue

    with open(pdf_path, "rb") as f:
        extracted = extractor.extract(f.read(), f"{doc_id}.pdf")

    gt_title = normalize(row["title_gt"])
    gt_abstract = normalize(row["abstract_gt"])
    gt_keywords = normalize(row["keyword_gt"])

    ex_title = normalize(extracted.title)
    ex_abstract = normalize(extracted.abstract)
    ex_keywords = normalize(extracted.keywords)

    results.append({
        "id": doc_id,

        "wer_title": wer(gt_title, ex_title),
        "cer_title": cer(gt_title, ex_title),

        "wer_abstract": wer(gt_abstract, ex_abstract),
        "cer_abstract": cer(gt_abstract, ex_abstract),

        "wer_keywords": wer(gt_keywords, ex_keywords),
        "cer_keywords": cer(gt_keywords, ex_keywords),
    })

In [14]:
df_results = pd.DataFrame(results)
df_results.head()

,id,wer_title,cer_title,wer_abstract,cer_abstract,wer_keywords,cer_keywords
0,1,1.0,0.886792,1.0,1.0,0.000000,0.000000
1,2,1.0,0.752941,1.0,1.0,0.000000,0.000000
2,3,1.0,0.959184,1.0,1.0,1.000000,1.000000
3,4,1.0,0.858639,1.0,1.0,1.083333,0.811321
4,5,1.0,0.861789,1.0,1.0,1.250000,0.845361


In [15]:
summary = df_results.mean(numeric_only=True).to_frame("average_score")
summary

,average_score
wer_title,0.836162
cer_title,0.722342
wer_abstract,0.866978
cer_abstract,0.839063
wer_keywords,0.831203
cer_keywords,1.564886


In [16]:
OUTPUT_PATH = "/content/drive/MyDrive/wer_cer_results.xlsx"

with pd.ExcelWriter(OUTPUT_PATH) as writer:
    df_results.to_excel(writer, sheet_name="per_document", index=False)
    summary.to_excel(writer, sheet_name="summary")

OUTPUT_PATH

'/content/drive/MyDrive/wer_cer_results.xlsx'

In [20]:
detailed_results = []

for _, row in df_gt.iterrows():
    doc_id = row["id"]
    pdf_path = os.path.join(PDF_FOLDER, f"{doc_id}.pdf")

    if not os.path.exists(pdf_path):
        continue

    with open(pdf_path, "rb") as f:
        extracted = extractor.extract(f.read(), f"{doc_id}.pdf")

    detailed_results.append({
        "id": doc_id,

        # Ground truth
        "gt_title": row["title_gt"],
        "gt_abstract": row["abstract_gt"],
        "gt_keywords": row["keyword_gt"],

        # Extracted
        "ex_title": extracted.title,
        "ex_abstract": extracted.abstract,
        "ex_keywords": extracted.keywords,
    })


In [21]:
df_compare = pd.DataFrame(detailed_results)
df_compare.head()

,id,gt_title,gt_abstract,gt_keywords,ex_title,ex_abstract,ex_keywords
0,1,"Cancer Incidence, Mortality, Years of Life Los...","IMPORTANCE The Global Burden of Diseases, Inju...",NaN,JAMAOncology | OriginalInvestigation,,
1,2,Antibacterial Pathways in Transition Metal-Bas...,"Across the planet, outbreaks of bacterial illn...","transition metals, nanocomposites, photodynami...",International Journal of Nanomedicine,,"transition, metals, nanocomposites, photodynam..."
2,3,Hypertension is associated with antibody respo...,Several types of vaccines have been developed ...,"COVID-19, Inactivated viral vaccine, Antibody ...",Vaccine40(2022)4046–4056,,
3,4,Preparation of antibacterial Gel/PCL nanofiber...,"In this work, blend nanofibrous scaffolds were...","Bone tissue engineering, DCPD-reinforced, Cont...",InorganicChemistryCommunications139(2022)109336,,"In, this, work, blend, nanofibrous, scaffolds,..."
4,5,"Occurrence, potential sources and ecological r...",Extensive global plastic production has led to...,"Microplastic, Marine ecosystem, Surface water,...",MarinePollutionBulletin174(2022)113282,,"Extensive, global, plastic, production, has, l..."


In [22]:
df_compare_short = df_compare[[
    "id",
    "gt_title", "ex_title",
    "gt_abstract", "ex_abstract",
    "gt_keywords", "ex_keywords"
]]

df_compare_short["gt_abstract"] = df_compare_short["gt_abstract"].str.slice(0, 200)
df_compare_short["ex_abstract"] = df_compare_short["ex_abstract"].str.slice(0, 200)

In [23]:
COMPARE_OUTPUT = "/content/gt_vs_extraction.xlsx"
df_compare.to_excel(COMPARE_OUTPUT, index=False)
COMPARE_OUTPUT


'/content/gt_vs_extraction.xlsx'